# 07 - Modelo CNN (ResNet18 + clasificador)

Los modelos clásicos (notebook 06) solo ven 5 propiedades globales de la imagen y casi no
superan al Dummy. Aquí cada frame pasa por una ResNet18 preentrenada en ImageNet
(`src/cnn_features.py`), que lo resume en un vector de 512 números, y sobre esos vectores se
entrena un clasificador. La evaluación es la misma del 06, para que los resultados se puedan
comparar directamente.

## 1. Configuración y carga

In [ ]:
import sys
import time
from pathlib import Path

import cv2
import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import GridSearchCV, GroupKFold, LeaveOneGroupOut
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

sys.path.insert(0, "..")
from src.cnn_features import N_FEATURES, extract_cnn_features_batch

RANDOM_STATE = 42
CLASES = ["bg", "challenge", "play", "throwin"]

ROOT = Path("..")
MODELS_DIR = ROOT / "models"
RESULTS_DIR = ROOT / "results"
CACHE = ROOT / "data" / "processed" / "cnn_features.npz"
MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

# mismos frames y mismo orden que usaron los modelos clásicos
frames = pd.read_csv(ROOT / "data" / "processed" / "frame_metrics.csv")
faltan = [p for p in frames["path"] if not (ROOT / p).exists()]
assert not faltan, (f"Faltan {len(faltan)} de {len(frames)} frames (p.ej. {faltan[0]}). "
                    "Baja los videos y corre el notebook 01 (ver README).")
print(f"{len(frames)} frames de {frames['video_id'].nunique()} videos")

9133 frames de 3 videos


## 2. Extracción de características con la ResNet18

Se pasa cada frame por la red y se guarda el vector en data/processed/cnn_features.npz
para no repetir este paso (no se versiona; se regenera en un par de minutos).

In [ ]:
BATCH = 64

if CACHE.exists() and list(np.load(CACHE, allow_pickle=True)["path"]) == list(frames["path"]):
    X = np.load(CACHE)["X"]
    print("cargado de", CACHE)
else:
    t0 = time.perf_counter()
    bloques = []
    for i in tqdm(range(0, len(frames), BATCH)):
        imgs = [cv2.imread(str(ROOT / p)) for p in frames["path"].iloc[i:i + BATCH]]
        bloques.append(extract_cnn_features_batch(imgs))
    X = np.vstack(bloques)
    extract_time = time.perf_counter() - t0
    np.savez_compressed(CACHE, X=X, path=frames["path"].to_numpy())
    print(f"extraídos en {extract_time:.0f} s ({extract_time * 1000 / len(X):.1f} ms/frame)")

y, groups = frames["label"], frames["video_id"]
assert X.shape == (len(frames), N_FEATURES)
X.shape

  0%|          | 0/143 [00:00<?, ?it/s]

extraídos en 243 s (26.6 ms/frame)


(9133, 512)

## 3. Cómo se evalúa

Igual que en el 06: leave-one-video-out. El clasificador se entrena 3 veces, dejando cada
vez un video completo fuera para probar, y la búsqueda de parámetros solo usa los videos de
entrenamiento. El tiempo de predicción **no** incluye la extracción con la ResNet (se reporta
aparte arriba).

In [ ]:
def evaluar(nombre, pipeline):
    """Evalúa con leave-one-video-out, guarda el modelo final y devuelve (métricas, predicciones)."""
    filas_metricas, filas_preds = [], []

    for train_idx, test_idx in LeaveOneGroupOut().split(X, y, groups):
        modelo = clone(pipeline)
        X_tr, y_tr = X[train_idx], y.iloc[train_idx]
        X_te, y_te = X[test_idx], y.iloc[test_idx]

        t0 = time.perf_counter()
        modelo.fit(X_tr, y_tr, groups=groups.iloc[train_idx])
        train_time = time.perf_counter() - t0
        fold = groups.iloc[test_idx].iloc[0]
        TUNING.append({"model": nombre, "fold": fold,
                       "best_params": modelo.best_params_,
                       "best_inner_f1": modelo.best_score_,
                       "n_combinaciones": len(modelo.cv_results_["params"]),
                       "tiempo_busqueda_s": train_time})

        t0 = time.perf_counter()
        pred = modelo.predict(X_te)
        proba_raw = modelo.predict_proba(X_te)
        pred_time_ms = (time.perf_counter() - t0) * 1000 / len(X_te)

        # columnas de probabilidad siempre en el orden de CLASES
        proba = pd.DataFrame(0.0, index=range(len(X_te)), columns=CLASES)
        proba[list(modelo.classes_)] = proba_raw

        f1_clase = f1_score(y_te, pred, labels=CLASES, average=None, zero_division=0)
        filas_metricas.append({
            "model": nombre,
            "fold": fold,
            "accuracy": accuracy_score(y_te, pred),
            "f1_macro": f1_score(y_te, pred, labels=CLASES, average="macro", zero_division=0),
            **{f"f1_{c}": f for c, f in zip(CLASES, f1_clase)},
            "train_time_s": train_time,
            "pred_time_ms": pred_time_ms,
        })

        filas_preds.append(pd.DataFrame({
            "video_id": frames["video_id"].iloc[test_idx].values,
            "time_s": frames["time_s"].iloc[test_idx].values,
            "label": y_te.values,
            "model": nombre,
            "pred": pred,
            **{f"proba_{c}": proba[c].values for c in CLASES},
        }))

    # modelo final con todos los datos, es el que usa la app
    t0 = time.perf_counter()
    busqueda = clone(pipeline).fit(X, y, groups=groups)
    TUNING.append({"model": nombre, "fold": "final",
                   "best_params": busqueda.best_params_,
                   "best_inner_f1": busqueda.best_score_,
                   "n_combinaciones": len(busqueda.cv_results_["params"]),
                   "tiempo_busqueda_s": time.perf_counter() - t0})
    joblib.dump(busqueda.best_estimator_, MODELS_DIR / f"{nombre}.joblib", compress=3)

    metricas = pd.DataFrame(filas_metricas)[[
        "model", "fold", "accuracy", "f1_macro", "f1_bg", "f1_play",
        "f1_challenge", "f1_throwin", "train_time_s", "pred_time_ms",
    ]]
    return metricas, pd.concat(filas_preds, ignore_index=True)

## 4. Clasificador sobre los vectores de la ResNet

Se prueban dos clasificadores y la búsqueda se queda con el mejor en F1 macro:

- **Regresión Logística**: cuánta regularización (`C`) y si se balancean las clases.
- **MLP** (una capa oculta): cuántas neuronas y cuánta regularización (`alpha`). Usa
  *early stopping* para no sobreajustar con tan pocos videos.

In [ ]:
TUNING = []

busqueda_cnn = GridSearchCV(
    Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression())]),
    param_grid=[
        {"clf": [LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)],
         "clf__C": [0.001, 0.01, 0.1, 1],
         "clf__class_weight": [None, "balanced"]},
        {"clf": [MLPClassifier(early_stopping=True, max_iter=300, random_state=RANDOM_STATE)],
         "clf__hidden_layer_sizes": [(128,), (256,)],
         "clf__alpha": [1e-4, 1e-2, 1]},
    ],
    scoring="f1_macro", cv=GroupKFold(n_splits=2), n_jobs=-1,
)

metricas_cnn, preds_cnn = evaluar("cnn", busqueda_cnn)
metricas_cnn

,model,fold,accuracy,f1_macro,f1_bg,f1_play,f1_challenge,f1_throwin,train_time_s,pred_time_ms
0,cnn,1606b0e6_0,0.649576,0.218179,0.072072,0.800644,0.000000,0.0,28.959405,0.017955
1,cnn,1606b0e6_1,0.733505,0.277290,0.216535,0.848000,0.044625,0.0,7.032781,0.005598
2,cnn,35bd9041_0,0.685286,0.254225,0.073620,0.819337,0.123944,0.0,6.869776,0.005895


## 5. Guardar resultados

In [ ]:
metricas_cnn.to_csv(RESULTS_DIR / "metrics_cnn.csv", index=False)
preds_cnn.to_csv(RESULTS_DIR / "predictions_cnn.csv", index=False)
tuning = pd.DataFrame(TUNING)
tuning["best_params"] = tuning["best_params"].astype(str)
tuning.to_csv(RESULTS_DIR / "tuning_cnn.csv", index=False)
print(f"metrics: {metricas_cnn.shape}, predictions: {preds_cnn.shape}")
tuning

metrics: (3, 10), predictions: (9133, 9)


,model,fold,best_params,best_inner_f1,n_combinaciones,tiempo_busqueda_s
0,cnn,1606b0e6_0,"{'clf': LogisticRegression(max_iter=2000, rand...",0.223712,14,28.959405
1,cnn,1606b0e6_1,"{'clf': MLPClassifier(early_stopping=True, max...",0.237097,14,7.032781
2,cnn,35bd9041_0,"{'clf': MLPClassifier(early_stopping=True, max...",0.257740,14,6.869776
3,cnn,final,"{'clf': MLPClassifier(early_stopping=True, max...",0.246795,14,15.767885


## 6. Comparación con los modelos clásicos (promedio ± desviación de los 3 videos)

In [ ]:
metrics = pd.concat([pd.read_csv(RESULTS_DIR / "metrics_clasicos.csv"), metricas_cnn],
                    ignore_index=True)
cols = ["accuracy", "f1_macro", "f1_bg", "f1_play", "f1_challenge", "f1_throwin",
        "train_time_s", "pred_time_ms"]
resumen = metrics.groupby("model")[cols].agg(["mean", "std"])
resumen.columns = [f"{c}_{a}" for c, a in resumen.columns]
resumen = resumen.sort_values("f1_macro_mean", ascending=False).reset_index()

tabla = resumen[["model"]].copy()
for c in cols[:6]:
    tabla[c] = resumen[f"{c}_mean"].map("{:.3f}".format) + " ± " + resumen[f"{c}_std"].map("{:.3f}".format)
tabla["train_time_s"] = resumen["train_time_s_mean"].round(2)
tabla["pred_time_ms"] = resumen["pred_time_ms_mean"].round(4)
tabla

,model,accuracy,f1_macro,f1_bg,f1_play,f1_challenge,f1_throwin,train_time_s,pred_time_ms
0,cnn,0.689 ± 0.042,0.250 ± 0.030,0.121 ± 0.083,0.823 ± 0.024,0.056 ± 0.063,0.000 ± 0.000,14.29,0.0098
1,rf,0.735 ± 0.076,0.240 ± 0.035,0.100 ± 0.118,0.850 ± 0.044,0.011 ± 0.018,0.000 ± 0.000,10.44,0.0143
2,hgb,0.620 ± 0.189,0.224 ± 0.002,0.089 ± 0.082,0.757 ± 0.153,0.051 ± 0.089,0.000 ± 0.000,38.18,0.0146
3,logreg,0.719 ± 0.090,0.221 ± 0.008,0.042 ± 0.074,0.840 ± 0.054,0.000 ± 0.000,0.000 ± 0.000,0.53,0.0003
4,dummy,0.780 ± 0.040,0.219 ± 0.006,0.000 ± 0.000,0.876 ± 0.025,0.000 ± 0.000,0.000 ± 0.000,0.00,0.0000


In [ ]:
# matriz de confusión del CNN sumando los 3 videos de prueba
pd.crosstab(preds_cnn["label"], preds_cnn["pred"], rownames=["real"], colnames=["predicho"])

predicho,bg,challenge,play,throwin
real,,,,
bg,99,18,539,1
challenge,137,33,881,0
play,803,135,6198,6
throwin,19,1,263,0


## 7. Conclusiones

**El CNN queda primero, pero por poco y sin una ventaja clara.** Tiene el F1 macro promedio más
alto (0.250 ± 0.030), por encima del Random Forest (0.240) y del Dummy (0.219). Pero la ventaja
sale de un solo video: en `1606b0e6_0` y `1606b0e6_1` empata con el Random Forest (0.218 vs 0.218
y 0.277 vs 0.280) y solo en `35bd9041_0` lo supera (0.254 vs 0.223). Con 3 videos de prueba esa
diferencia no alcanza para decir que un modelo es mejor que el otro.

**Las clases difíciles siguen sin resolverse:**

- `throwin`: F1 de 0 en los tres videos. El modelo predice `throwin` 7 veces en 9 133 frames y
  ninguna es correcta.
- `challenge`: es donde más mejora el CNN (F1 0.056 vs 0.011 del Random Forest), pero solo
  reconoce 33 de 1 051 (recall de 3 %).
- `play`: cuando el modelo dice `play` acierta el 78.6 % de las veces, prácticamente la proporción
  de `play` en los datos (78.2 %). No está distinguiendo los pases del resto.
- Su accuracy (0.689) es menor que la del Dummy (0.780): el CNN se arriesga a predecir otras
  clases y la mayoría de esas veces se equivoca (803 frames `play` los predice como `bg`).

**Lo que eligió la búsqueda.** En el modelo final ganó el MLP de 256 neuronas con `alpha = 1`; en
el primer video ganó la Regresión Logística con `C = 0.001`. En los dos casos es la regularización
más fuerte que se probó: el clasificador funciona mejor cuanto menos se ajusta a los videos de
entrenamiento, señal de que lo que aprende de un partido casi no sirve para otro. Nunca eligió
`class_weight="balanced"`.

**Costo.** Lo caro es la extracción: 26.6 ms por frame con la ResNet18 en CPU (4 min para los
9 133 frames), contra fracciones de milisegundo de las 5 propiedades de los modelos clásicos. Ya
con los vectores, entrenar (14 s) y predecir (0.01 ms por frame) cuesta lo mismo que el Random
Forest. En la app no se nota: clasificar una imagen tarda unas decenas de milisegundos.

**Por qué no funciona mejor:**

1. La ResNet18 aprendió con fotos de ImageNet, donde el objeto ocupa buena parte de la imagen.
   Aquí el frame de 1920×1080 se reduce a 256 px de alto y se recorta un cuadrado central de
   224×224: los jugadores (~18 px) quedan de unos 4 px y se pierde casi la mitad del ancho del
   campo. El vector describe la escena (césped, gradas, tipo de plano), no lo que hacen los
   jugadores.
2. Un frame suelto no tiene movimiento, y lo que distingue un pase de una disputa es justamente
   cómo se mueven los jugadores.
3. Las etiquetas tienen ruido: un frame 2 s antes de un pase se etiqueta `play` aunque se vea
   igual que uno `bg`.
4. Solo hay 3 videos (2 partidos) para entrenar y probar.

**En resumen**, las características de una CNN preentrenada describen la imagen mejor que las 5
propiedades globales y dan el mejor F1 macro del proyecto, pero la mejora es pequeña y no resuelve
el problema: con frames sueltos y a esta resolución, ningún modelo distingue de forma confiable
los tres eventos. Para avanzar haría falta usar la imagen sin recortar o recortes alrededor de la
jugada, modelos que vean varios frames seguidos (CNN 3D, redes de dos flujos o transformers de
video) y más partidos.